# CyberKnife — Module Tests

One section per module in the assignment. Each section explains the test, predicts the result, runs the module and
compares. All coordinates are in mm unless stated otherwise.

In [1]:
import sys
import numpy as np

sys.path.insert(0, "..")   # repo root, so the project modules can be imported

def fmt(uv):
    return "(%.2f, %.2f)" % tuple(np.round(uv, 2) + 0.0)   # + 0.0 avoids printing -0.00

## Part 1 — Frame Transforms

**Test required:** test each frame transform (MD ↔ CK, CK ↔ Detector A/B, Detector A/B ↔ Image A/B) on suitable points; explain the choice, predict the result, run the module and show it matches.

*Not filled in yet.*

## Part 2 — Marker Transforms

**Test required:** transform the three markers through the chain of frames, save the results, and inspect them visually.

*Not filled in yet.*

## Part 3 — Forward Projector

**Approach.** Each test point is chosen so that its projection can be predicted by hand and so that it checks one
specific property of the projector. A point is magnified by **M = SDD / (SAD − d) = 2000 / (1000 − d)**, where *d* is
how far the point lies from the isocentre towards that detector's X-ray source. Points level with the isocentre
(*d* = 0) are magnified exactly 2×, and points closer to the source are magnified more. The detector coordinates are then
**u = M × z** (height) and **v = M × sideways offset** along the detector's *v* axis. Detectors A and B sit at +45° and
−45°, so their beams are perpendicular; P3 uses this.

**Test points** (CK coordinates in mm, predictions as detector (u, v) in mm)

1. **P1 = (0, 0, 0), the isocentre.** Both central rays pass through it, so it must hit the centre of both detectors.
   This checks that each source, the isocentre and the detector centre are aligned.
   Predicted: A (0, 0), B (0, 0).
2. **P2 = (0, 0, 50), on the rotation axis.** It is equally far from both sources, so M = 2 on both, and it should move
   *up* the detector because +u is CK +z. This checks the magnification and the u direction.
   Predicted: A (100, 0), B (100, 0).
3. **P3 = (7.07, 7.07, 50), 10 mm along detector A's v axis.** For A it is level with the isocentre (M = 2), so v = +20,
   which checks the v direction. For B the same point lies on the central ray (v = 0) but 10 mm closer to the source,
   so M = 2000 / 990 = 2.02 and u = 101.01, which checks that magnification grows with depth.
   Predicted: A (100, 20), B (101.01, 0).

In [2]:
from projector.forward_projector import forward_project

# CK point (mm) and predicted (u, v) on detectors A and B
tests = {
    "P1": ([0, 0, 0],                              {"A": (0, 0),   "B": (0, 0)}),
    "P2": ([0, 0, 50],                             {"A": (100, 0), "B": (100, 0)}),
    "P3": ([10 / np.sqrt(2), 10 / np.sqrt(2), 50], {"A": (100, 20), "B": (50 * 2000 / 990, 0)}),
}

print(f"{'Point':<7}{'Detector':<10}{'Predicted (u, v)':<20}{'Projected (u, v)':<20}Match")
for name, (point, predicted) in tests.items():
    projected = forward_project(np.array(point, dtype=float))
    for det in ("A", "B"):
        u, v, w = projected[det]
        match = np.allclose((u, v), predicted[det]) and abs(w) < 1e-9   # w ≈ 0: the hit lies on the detector
        print(f"{name:<7}{det:<10}{fmt(predicted[det]):<20}{fmt((u, v)):<20}{'yes' if match else 'NO'}")

Point  Detector  Predicted (u, v)    Projected (u, v)    Match
P1     A         (0.00, 0.00)        (0.00, 0.00)        yes
P1     B         (0.00, 0.00)        (0.00, 0.00)        yes
P2     A         (100.00, 0.00)      (100.00, 0.00)      yes
P2     B         (100.00, 0.00)      (100.00, 0.00)      yes
P3     A         (100.00, 20.00)     (100.00, 20.00)     yes
P3     B         (101.01, 0.00)      (101.01, 0.00)      yes


**Result.** All six projections match the predictions, and every projected point lies on the detector plane (w ≈ 0).
The projector aligns both detectors with the isocentre, applies the 2× magnification, orients u and v as specified,
and magnifies points closer to the source more.

## Part 4 — Marker Projector

**Approach.** The assignment asks for a rough hand estimate of where each marker should land. All three markers lie
within about 50 mm of the isocentre, so the estimate ignores depth and uses **M ≈ 2**: **u ≈ 2 × z** and
**v ≈ 2 × sideways offset**. Detector A's v axis is (0.71, 0.71, 0) and B's is (0.71, −0.71, 0), so the sideways
offset is 0.71 × (x + y) on A and 0.71 × (x − y) on B. Ignoring depth keeps M between about 1.9 and 2.1, so the
projections should have the same signs as the estimates and agree to within a few mm (we allow 4 mm).

**Hand estimates** (detector (u, v) in mm)

| Marker (CK) | Estimate on A | Estimate on B |
|---|---|---|
| M1 = (−19, 28, 11) | (22, 12.7) | (22, −66.5) |
| M2 = (43, 14, 2) | (4, 80.6) | (4, 41.0) |
| M3 = (2, 48, −6) | (−12, 70.7) | (−12, −65.1) |

In [3]:
from projector.marker_projector import MARKERS_CK, project_markers

# Rough hand estimates (u, v) in mm, using M ≈ 2 and ignoring depth
estimates = {
    "M1": {"A": (22, 12.7),  "B": (22, -66.5)},
    "M2": {"A": (4, 80.6),   "B": (4, 41.0)},
    "M3": {"A": (-12, 70.7), "B": (-12, -65.1)},
}

results = project_markers(MARKERS_CK)

print(f"{'Marker':<8}{'Detector':<10}{'Estimate (u, v)':<18}{'Projected (u, v)':<20}{'Image (u, v) px':<22}{'Diff':<8}Close")
for name, est in estimates.items():
    for det in ("A", "B"):
        uv = results[name][f"Detector_{det}"][:2]
        image = results[name][f"Image_{det}"]
        diff = np.abs(uv - est[det]).max()
        close = diff < 4 and np.all(np.sign(uv) == np.sign(est[det]))
        print(f"{name:<8}{det:<10}{fmt(est[det]):<18}{fmt(uv):<20}{fmt(image):<22}{diff:<8.2f}{'yes' if close else 'NO'}")

Marker  Detector  Estimate (u, v)   Projected (u, v)    Image (u, v) px       Diff    Close
M1      A         (22.00, 12.70)    (22.76, 13.17)      (1227.56, 1131.65)    0.76    yes
M1      B         (22.00, -66.50)   (22.14, -66.89)     (1221.41, 331.06)     0.39    yes
M2      A         (4.00, 80.60)     (3.92, 78.99)       (1039.20, 1789.90)    1.61    yes
M2      B         (4.00, 41.00)     (4.17, 42.73)       (1041.68, 1427.35)    1.73    yes
M3      A         (-12.00, 70.70)   (-12.40, 73.09)     (875.97, 1730.88)     2.39    yes
M3      B         (-12.00, -65.10)  (-12.44, -67.44)    (875.60, 325.62)      2.34    yes


**Result.** Every marker lands on the estimated side of both detectors, at most 2.4 mm from the rough estimate, which is
the size of error expected from ignoring depth. For example, M3 sits 48 mm towards the ceiling, i.e. towards both
sources, so it is magnified slightly more than 2 on both detectors and lands a little further out than estimated.

## Part 5 — Digitally Reconstructed Radiographs

**Test required:** generate DRR pairs for the homogeneous sphere and for the vertebra with markers; describe what you expect to see, compare with what you observe, and explain.

*Not filled in yet.*

## Part 6 — Marker Localization

**Test required:** localize the marker centres in the DRRs; compare computed, visually picked and true positions, and explain the differences.

*Not filled in yet.*

## Part 7 — Marker Reconstruction

**Test required:** feed the known marker detector positions in and show the known CK positions come back with near-zero residual error (REM).

*Not filled in yet.*

## Part 8 — Marker Correspondences

**Test required:** show correct correspondences for the normal projections and with two markers swapped on one detector; repeat with a second method.

*Not filled in yet.*

## Part 9 — Marker Ambiguity

**Test required:** alter the marker positions to create an ambiguity, explain the cause, and show it is detected; repeat with a second method.

*Not filled in yet.*

## Part 10 — Marker Reconstruction Error Analysis

**Test required:** for MLE = 0, 1, 2, … mm over all 360 × 360 direction pairs, compute MRE and REM; plot MLE vs MRE and REM vs MRE, find maxMLE for MRE < 5 mm, and assess REM as a warning metric.

*Not filled in yet.*